In [1]:
%matplotlib inline

import torch
import torch.nn as nn

import tiktoken
import matplotlib.pyplot as plt


%load_ext autoreload
%autoreload 2

from gpt import GPTModel

In [2]:
# configuration of the small GPT-2
GPT_CONFIG_124M={ # GPT2-small
    "vocab_size":50257,       # Vocabulary size
    "context_length":1024,    # Context length
    "emb_dim":768,            # Embedding dimension
    "n_heads":12,             # Number of attention heads
    "n_layers":12,            # Number of layers
    "drop_rate":0.1,          # Dropout rate
    "qkv_bias": False         # Query-key-value bias
}

torch.manual_seed(123)
inputs=torch.tensor([
    [6109, 3626, 6100, 345],
    [6109, 1110, 6622, 257]
]
)

print(f"{inputs.shape=}, {inputs.dtype=}") # 2 input texts with 6 token each, each token is 3D embeddings
print(f"{inputs}")

model=GPTModel(GPT_CONFIG_124M)
out=model(inputs)
print(f"{out.shape=}\n{out}")

inputs.shape=torch.Size([2, 4]), inputs.dtype=torch.int64
tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])
out.shape=torch.Size([2, 4, 50257])
tensor([[[ 0.1381,  0.0077, -0.1963,  ..., -0.0222, -0.1060,  0.1717],
         [ 0.3865, -0.8408, -0.6564,  ..., -0.5163,  0.2369, -0.3357],
         [ 0.6989, -0.1829, -0.1631,  ...,  0.1472, -0.6504, -0.0056],
         [-0.4290,  0.1669, -0.1258,  ...,  1.1579,  0.5303, -0.5549]],

        [[ 0.1094, -0.2894, -0.1467,  ..., -0.0557,  0.2911, -0.2824],
         [ 0.0882, -0.3552, -0.3527,  ...,  1.2930,  0.0053,  0.1898],
         [ 0.6091,  0.4702, -0.4094,  ...,  0.7688,  0.3787, -0.1974],
         [-0.0612, -0.0737,  0.4751,  ...,  1.2463, -0.3834,  0.0609]]],
       grad_fn=<UnsafeViewBackward0>)


In [3]:
total_params=sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params}")
print(f"{model.tok_emb.weight.shape=}")
print(f"{model.out_head.weight.shape=}")
total_params_gpt2=(total_params -
                   sum(p.numel() for p in model.out_head.parameters()))
print(f"Number of trainable parameters after considering weight tying: {total_params_gpt2}")

Total number of parameters: 163009536
model.tok_emb.weight.shape=torch.Size([50257, 768])
model.out_head.weight.shape=torch.Size([50257, 768])
Number of trainable parameters after considering weight tying: 124412160


In [4]:
n_ffn_params=n_attn_params=0
for block in model.trf_blocks:
    n_attn_params+=sum(p.numel() for p in block.att.parameters())
    n_ffn_params+=sum(p.numel() for p in block.ff.parameters())
print(f"{n_attn_params=}, {n_ffn_params=}")

n_attn_params=28320768, n_ffn_params=56669184


In [5]:
# Calculate the total size in bytes (assuming float32, 4 bytes per parameter)
total_size_bytes=total_params*4
total_size_mb=total_size_bytes/(1024*1024) # convert to megabytes
print(f"Total size of the model: {total_size_mb:.2f} MB")

Total size of the model: 621.83 MB


In [6]:
def model_size(model):
    """ Compute number of parameters and model size in MB
    Args:
        model (nn.Module)
    Returns:
        (int): Number of total parameters
        (int): Number of parameters in multihead attention
        (int): Number of parameters in feedforward
        (float): Size of model in MB
    """
    
    n_total_params=sum(p.numel() for p in model.parameters())

    n_ffn_params=n_attn_params=0
    for block in model.trf_blocks:
        n_ffn_params+=sum(p.numel() for p in block.ff.parameters())
        n_attn_params+=sum(p.numel() for p in block.att.parameters())

    size_in_mb=(n_total_params*4)/(1024*1024)
    return n_total_params, n_attn_params, n_ffn_params, size_in_mb

In [7]:
# configuration of the small GPT-2
GPT_CONFIG_medium={
    "vocab_size":50257,       # Vocabulary size
    "context_length":1024,    # Context length
    "emb_dim":1024,            # Embedding dimension
    "n_heads":16,             # Number of attention heads
    "n_layers":24,            # Number of layers
    "drop_rate":0.1,          # Dropout rate
    "qkv_bias": False         # Query-key-value bias
}
GPT_CONFIG_large={
    "vocab_size":50257,       # Vocabulary size
    "context_length":1024,    # Context length
    "emb_dim":1280,            # Embedding dimension
    "n_heads":20,             # Number of attention heads
    "n_layers":36,            # Number of layers
    "drop_rate":0.1,          # Dropout rate
    "qkv_bias": False         # Query-key-value bias
}
GPT_CONFIG_xl={
    "vocab_size":50257,       # Vocabulary size
    "context_length":1024,    # Context length
    "emb_dim":1600,            # Embedding dimension
    "n_heads":25,             # Number of attention heads
    "n_layers":48,            # Number of layers
    "drop_rate":0.1,          # Dropout rate
    "qkv_bias": False         # Query-key-value bias
}

for name, cfg in zip(['medium', 'large', 'xl'],
                    [GPT_CONFIG_medium, GPT_CONFIG_large, GPT_CONFIG_xl]):
    model=GPTModel(cfg)
    n_total_params, n_attn_params, n_ffn_params, size_in_mb=model_size(model)
    print(f"{name}: total {n_total_params}, attn {n_attn_params}, ffn {n_ffn_params}, size {size_in_mb} MB ")

medium: total 406212608, attn 100687872, ffn 201449472, size 1549.578125 MB 
large: total 838220800, attn 235975680, ffn 472089600, size 3197.55859375 MB 
xl: total 1637792000, attn 491596800, ffn 983424000, size 6247.6806640625 MB 


In [12]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    """
    Args:
        idx (torch.Tensor): Current context indices of size (batch, n_tokens)
        max_new_tokens (int): Maximum number of new tokens to be generated
        
    """
    for _ in range(max_new_tokens):
        # crop current context if it exceeds the supported context size, e.g., if LLM supports only 5 tokens, 
        # and the context size is 10, then only the last 5 tokens are used as context
        idx_cond=idx[:, -context_size:]
        with torch.no_grad(): logits=model(idx_cond) # (batch, n_tokens, vocab_size)

        # focus only on the last time step so (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits=logits[:,-1] # (batch, vocab_size)
        probs=torch.softmax(logits, dim=-1) # (batch, vocab_size)
        idx_next=torch.argmax(probs, dim=-1, keepdim=True) # (batch, 1)
        idx=torch.cat((idx, idx_next), dim=1) # (batch, n_tokens+1,2,3,..max_new_tokens)
    return idx


tokenizer=tiktoken.get_encoding('gpt2')
start_context="Hello, I am"
encoded=tokenizer.encode(start_context)
print(f'encoded: {encoded}')
encoded_tensor=torch.tensor(encoded).unsqueeze(0) # add batch dimension
print(f"{encoded_tensor.shape=}")

model.eval() # disable dropout
out=generate_text_simple(model=model, idx=encoded_tensor, max_new_tokens=6, context_size=GPT_CONFIG_124M['context_length'])
print(f"{out.shape=}\n{out}")
decoded_text=tokenizer.decode(out.squeeze(0).tolist())
print(f"{decoded_text=}")

encoded: [15496, 11, 314, 716]
encoded_tensor.shape=torch.Size([1, 4])
out.shape=torch.Size([1, 10])
tensor([[15496,    11,   314,   716, 24698, 46754, 47115, 48663, 16027, 32143]])
decoded_text='Hello, I am Biology franc Inspection Aki Legal starvation'


In [9]:
GPT_CONFIG_124M['context_length']

1024